In [2]:
# CREATE TABLE fact_stint (
#     stint_id bigserial primary key,
#     session_id int not null,
#     driver_id int not null,
#     compound_id int not null,
#     stint_number int,
#     start_lap int,
#     end_lap int,
#     laps_count int,
#     avg_lap_time_s numeric,
#     deg_rate_s_per_lap numeric,

#     CONSTRAINT fk_stint_session  FOREIGN KEY (session_id)  REFERENCES dim_session(session_id),
#     CONSTRAINT fk_stint_driver   FOREIGN KEY (driver_id)   REFERENCES dim_driver(driver_id),
#     CONSTRAINT fk_stint_compound FOREIGN KEY (compound_id) REFERENCES dim_compound(compound_id)
# );
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import fastf1

fastf1.Cache.enable_cache('/tmp/fastf1_cache')

spark = SparkSession.builder \
        .appName("F1_Telemetry_Processing") \
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2") \
        .getOrCreate()

url = "jdbc:postgresql://localhost:5432/telemetry"
properties = {"user": "michal", "password": "root", "driver": "org.postgresql.Driver"}

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/31 14:09:12 WARN Utils: Your hostname, michal-MS-7996, resolves to a loopback address: 127.0.1.1; using 192.168.0.58 instead (on interface enp2s0)
26/03/31 14:09:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/michal/.ivy2.5.2/cache
The jars for the packages stored in: /home/michal/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a6a9789d-f3be-4221-a744-2dcdfe7cd5aa;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.2 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 125ms :: artifacts dl 7ms
	:: modules in use:
	org.checkerfram

In [3]:
dim_season = spark.read.jdbc(url=url, table='dim_season', properties=properties)
dim_season_df = dim_season.collect()
dim_session = spark.read.jdbc(url=url, table='dim_session', properties=properties)
dim_session_df = dim_session.collect()
dim_driver = spark.read.jdbc(url=url, table='dim_driver', properties=properties)
dim_driver_df = dim_driver.collect()
dim_event = spark.read.jdbc(url=url, table='dim_event', properties=properties)
dim_event_df = dim_event.collect()
dim_compound = spark.read.jdbc(url=url, table='dim_compound', properties=properties)
dim_compound_df = dim_compound.collect()

season_map = {row['season_id']: row['year'] for row in dim_season_df}
driver_map = {row['code']: row['driver_id'] for row in dim_driver_df}
event_map = {
    row["event_id"]: (row["season_id"], row["round_number"])
    for row in dim_event.collect()
}
compound_map = {row['name']: row['compound_id'] for row in dim_compound_df}

In [22]:
def build_fact_stint(session, session_id, driver_map, compound_map, spark):
    laps = session.laps

    for col in laps.columns:
        if pd.api.types.is_timedelta64_dtype(laps[col]):
            laps[col] = laps[col].dt.total_seconds()

    stints = laps.groupby(["Driver", "Stint", "Compound"]).agg(
        start_lap = ("LapNumber", "min"),
        end_lap = ("LapNumber", "max"),
        laps_count = ("LapNumber", "count"),
        avg_lap_time_s = ("LapTime",   "mean"),
    ).reset_index()

   # Degradacja
    accurate  = laps[laps["IsAccurate"] == True].sort_values("LapNumber")

    deg_agg = accurate.groupby(["Driver", "Stint"]).agg(
        first_lap = ("LapTime", "first"),
        last_lap  = ("LapTime", "last"),
        count     = ("LapTime", "count"),
    ).reset_index()
    
    deg_agg["deg_rate_s_per_lap"] = (
        (deg_agg["last_lap"] - deg_agg["first_lap"]) / deg_agg["count"]
    )
    
    # Zastąp None dla stintów z < 3 okrążeń
    deg_agg.loc[deg_agg["count"] < 3, "deg_rate_s_per_lap"] = None
    
    deg = deg_agg[["Driver", "Stint", "deg_rate_s_per_lap"]]
    
    stints = stints.merge(deg, on=["Driver", "Stint"], how="left")

    stints_df = spark.createDataFrame(stints)
    stints_df = stints_df.withColumn("session_id", F.lit(session_id))

    driver_mapping   = F.create_map([F.lit(x) for pair in driver_map.items()   for x in pair])
    compound_mapping = F.create_map([F.lit(x) for pair in compound_map.items() for x in pair])

    stints_df = stints_df.withColumn("driver_id",   driver_mapping[F.col("Driver")])
    stints_df = stints_df.withColumn("compound_id", compound_mapping[F.col("Compound")])

    # drop wierszy jeżeli compound albo kierowca null
    stints_df = stints_df.filter(F.col("driver_id").isNotNull())
    stints_df = stints_df.filter(F.col("compound_id").isNotNull())
    
    stints_df = stints_df.select([
        "session_id", "driver_id", "compound_id",
        "Stint", "start_lap", "end_lap", "laps_count",
        "avg_lap_time_s", "deg_rate_s_per_lap",
    ])

    stints_df = stints_df.withColumnRenamed("Stint", "stint_number")

    # NaN -> null
    float_cols = [f.name for f in stints_df.schema.fields
                  if isinstance(f.dataType, (DoubleType, FloatType))]
    for col in float_cols:
        stints_df = stints_df.withColumn(col,
            F.when(F.isnan(F.col(col)), None).otherwise(F.col(col)))

    stints_df = stints_df \
        .withColumn("driver_id",    F.expr("try_cast(driver_id    as INT)")) \
        .withColumn("compound_id",  F.expr("try_cast(compound_id  as INT)")) \
        .withColumn("stint_number", F.expr("try_cast(stint_number as INT)")) \
        .withColumn("start_lap",    F.expr("try_cast(start_lap    as INT)")) \
        .withColumn("end_lap",      F.expr("try_cast(end_lap      as INT)")) \
        .withColumn("laps_count",   F.expr("try_cast(laps_count   as INT)"))

    return stints_df

In [23]:
loaded = 0
errors = 0
for row in dim_session_df:
    season_id, round_number = event_map[row.event_id]

    try:
        session = fastf1.get_session(season_map[season_id], round_number, row.session_type)
        session.load(laps=True, telemetry=False, weather=False, messages=False)
    except Exception as e:
        print(f"Failed: {season_map[season_id]} R{round_number} {row.session_type}: {e}")
        errors += 1
        continue

    sdf = build_fact_stint(session, row.session_id, driver_map, compound_map, spark)
    sdf.write.jdbc(url=url, table='fact_stint', mode='append', properties=properties)
    loaded += 1

print(f'loaded {loaded/(loaded+errors)*100}% ({loaded}, {errors})')

core           INFO 	Loading data for Austrian Grand Prix - Practice 1 [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['1', '2', '3', '4', '10', '11', '14', '16', '18', '20', '22', '23', '24', '27', '31', '44', '55', '63', '77', '81']
core           INFO 	Loading data for Austrian Grand Prix - Sprint Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core        WARNING 	Sprint Qualifying is not supported by Ergast! Limited results are calculated from timing data.
req            INFO 	Usi

Failed: 2025 R20 FP1: any API: 500 calls/h


core           INFO 	Loading data for Mexico City Grand Prix - Practice 3 [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


Failed: 2025 R20 FP2: any API: 500 calls/h


core           INFO 	Loading data for Mexico City Grand Prix - Qualifying [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


...

Failed: 2024 R9 FP2: any API: 500 calls/h


core           INFO 	Loading data for Canadian Grand Prix - Qualifying [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


Failed: 2024 R9 FP3: any API: 500 calls/h


core           INFO 	Loading data for Canadian Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


Failed: 2024 R9 Q: any API: 500 calls/h
Failed: 2024 R9 R: any API: 500 calls/h
Failed: 2022 R15 FP1: any API: 500 calls/h
Failed: 2022 R15 FP2: any API: 500 calls/h
Failed: 2022 R15 FP3: any API: 500 calls/h
Failed: 2022 R15 Q: any API: 500 calls/h
Failed: 2022 R15 R: any API: 500 calls/h
Failed: 2022 R16 FP1: any API: 500 calls/h
Failed: 2022 R16 FP2: any API: 500 calls/h
Failed: 2022 R16 FP3: any API: 500 calls/h
Failed: 2022 R16 Q: any API: 500 calls/h
Failed: 2022 R16 R: any API: 500 calls/h
loaded 28.815789473684212% (219, 541)


In [24]:
#~57k rekordów